In [111]:
import pandas as pd
from pathlib import Path

In [112]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
raw_data_dir = ROOT / "data" / "raw"

In [113]:
competitions = pd.read_csv(raw_data_dir / "competitions.csv.gz")
clubs = pd.read_csv(raw_data_dir / "clubs.csv.gz")
valuations = pd.read_csv(raw_data_dir / "player_valuations.csv.gz")
players = pd.read_csv(raw_data_dir / "players.csv.gz")
appearances = pd.read_csv(raw_data_dir / "appearances.csv.gz")

In [114]:
pl_clubs = clubs[(clubs["domestic_competition_id"] == "GB1") & (clubs["last_season"] == 2025)]

pl_players = players[
    players["current_club_id"].isin(pl_clubs["club_id"]) & (players["last_season"] == 2025)
].merge(pl_clubs[["club_id", "name"]].rename(columns={"name": "club"}), left_on="current_club_id", right_on="club_id")

pl_appearances = appearances.merge(pl_players[["player_id", "name", "club"]], on="player_id")

## Data quality checks

Each check counts the number of problematic rows (0 = OK).
The scope of the analysis is limited to the Premier League 2025/2026 season.

In [115]:
all_player_ids = set(players["player_id"]) 
all_valuation_pids = valuations["player_id"]

competitions = competitions[competitions["competition_id"] == "GB1"]
clubs = clubs[(clubs["domestic_competition_id"] == "GB1") & (clubs["last_season"] == 2025)]
appearances = appearances[(appearances["competition_id"] == "GB1") & (appearances["date"] >= "2025-07-01")]
players = players[players["current_club_id"].isin(clubs["club_id"]) & (players["last_season"] == 2025)]
valuations = valuations[valuations["player_id"].isin(players["player_id"])]

results = []

def check(table, description, n_bad, n_total):
    n_bad = pd.Series(n_bad).sum()
    results.append({"table": table, "check": description, "n_bad": int(n_bad),
                    "pct": round(100 * n_bad / n_total, 2), "status": "OK" if n_bad == 0 else "CHECK"})


for name, df, key in [("competitions", competitions, "competition_id"), ("clubs", clubs, "club_id"),
                      ("players", players, "player_id"), ("appearances", appearances, "appearance_id")]:
    
    check(name, "duplicated rows", df.duplicated().sum(), len(df))
    check(name, f"duplicated {key}", df[key].duplicated().sum(), len(df))

check("player_valuations", "duplicated rows", valuations.duplicated().sum(), len(valuations))

check("player_valuations", "player_id not in players", ~all_valuation_pids.isin(all_player_ids), len(all_valuation_pids))
check("appearances", "player_id not in players", ~appearances["player_id"].isin(all_player_ids), len(appearances))
check("clubs", "domestic_competition_id not in competitions", ~clubs["domestic_competition_id"].isin(competitions["competition_id"]), len(clubs))

check("players", "position missing / 'Missing'", players["position"].isna() | (players["position"] == "Missing"), len(players))
check("players", "market_value_in_eur < 0", players["market_value_in_eur"] < 0, len(players))
check("players", "height_in_cm outside [150, 220]", ~players["height_in_cm"].dropna().between(150, 220), players["height_in_cm"].notna().sum())
check("player_valuations", "market_value_in_eur <= 0", valuations["market_value_in_eur"] <= 0, len(valuations))
check("appearances", "minutes_played outside [0, 90]", ~appearances["minutes_played"].between(0, 90), len(appearances))
check("appearances", "negative goals / assists / cards",
      (appearances[["goals", "assists", "yellow_cards", "red_cards"]] < 0).any(axis=1), len(appearances))

# 4. Dates
dob = pd.to_datetime(players["date_of_birth"], errors="coerce")
check("players", "age outside [14, 50] (valued players)", ~((pd.Timestamp.today() - dob).dt.days / 365.25).dropna().between(14, 50), dob.notna().sum())
for name, df in [("player_valuations", valuations), ("appearances", appearances)]:
    d = pd.to_datetime(df["date"], errors="coerce")
    check(name, "date unparsable or in the future", d.isna() | (d > pd.Timestamp.today()), len(df))

pd.DataFrame(results)

,table,check,n_bad,pct,status
0,competitions,duplicated rows,0,0.0,OK
1,competitions,duplicated competition_id,0,0.0,OK
2,clubs,duplicated rows,0,0.0,OK
3,clubs,duplicated club_id,0,0.0,OK
4,players,duplicated rows,0,0.0,OK
5,players,duplicated player_id,0,0.0,OK
6,appearances,duplicated rows,0,0.0,OK
7,appearances,duplicated appearance_id,0,0.0,OK
8,player_valuations,duplicated rows,0,0.0,OK
9,player_valuations,player_id not in players,0,0.0,OK
